In [1]:
#imports
import json, time, requests, duckdb, re, random
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from pytrends.request import TrendReq
import matplotlib.pyplot as plt
import yfinance as yf

In [2]:
#scrapes full S&P500 companies list from Wikipedia
path=Path("../../QSS20_FA26/finalproject/data/s&p500list.html")
path.parent.mkdir(parents=True,exist_ok=True)
session=requests.Session()
session.headers.update({"User-Agent": "Dyanne Jiang dyanne.y.jiang.30@dartmouth.edu"})
if path.exists():
    print("Already saved.")
else:
    res1=session.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",timeout=30)
    path.write_text(res1.text,encoding="utf-8")

Already saved.


In [3]:
#parses raw full S&P500 companies list to leave only a dataframe of company name, ticker, sector, and CIK
soup=BeautifulSoup(path.read_text(encoding="utf-8"),"html.parser")
for s in soup.select("sup.reference"):
    s.decompose()
sp500=soup.select_one("table#constituents")
sp500list=[]
for row in sp500.find_all("tr"):
    if row.find("td"):
        cells=row.find_all(["th","td"])
        sp500list.append({"Company Name":cells[1].get_text(" ",strip=True),"Ticker":cells[0].get_text(" ",strip=True),"Sector":cells[2].get_text(" ",strip=True),"CIK":cells[6].get_text(" ",strip=True).zfill(10)})
sp500df=pd.DataFrame(sp500list)
sp500df

,Company Name,Ticker,Sector,CIK
0,3M,MMM,Industrials,0000066740
1,A. O. Smith,AOS,Industrials,0000091142
2,Abbott Laboratories,ABT,Health Care,0000001800
3,AbbVie,ABBV,Health Care,0001551152
4,Accenture,ACN,Information Technology,0001467373
...,...,...,...,...
498,Xylem Inc.,XYL,Industrials,0001524472
499,Yum! Brands,YUM,Consumer Discretionary,0001041061
500,Zebra Technologies,ZBRA,Information Technology,0000877212
501,Zimmer Biomet,ZBH,Health Care,0001136869


In [4]:
#scrapes the 10-K reports of each company from SEC EDGAR
url=[]
for cik in sp500df["CIK"]:
    res2=session.get(f"https://data.sec.gov/submissions/CIK{cik}.json",timeout=30)
    time.sleep(0.15)
    recent=res2.json()["filings"]["recent"]
    if res2.status_code!=200 or "10-K" not in recent["form"]:
        url.append(None)
        continue
    i=recent["form"].index("10-K")
    acc=recent["accessionNumber"][i].replace("-","")
    doc=recent["primaryDocument"][i]
    url.append(f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc}/{doc}")
sp500df["url"]=url
sp500df=sp500df.drop_duplicates(subset="CIK").reset_index(drop=True)

In [5]:
#parses the Item 1 company description from each 10-K report
descriptions=[]
for url in sp500df["url"]:
    if pd.isna(url):
        descriptions.append(None)
        continue
    res10k=session.get(url,timeout=60)
    time.sleep(0.15)
    fulltext=BeautifulSoup(res10k.text,"html.parser").get_text(" ")
    fulltext=re.sub(r"\s+"," ",fulltext.replace("\xa0"," "))
    start=[m.start() for m in re.finditer(r"item\s*1\s*[\.:\-–—]?\s*business",fulltext,re.I)]
    end=[m.start() for m in re.finditer(r"item\s*1a\s*[\.:\-–—]?\s*risk\s*factors",fulltext,re.I)]
    longest=""
    for s in start:
        endfound=None
        for e in end:
            if e>s:
                endfound=e
                break
        if endfound is not None:
            section=fulltext[s:endfound]
            if len(section)>len(longest):
                longest=section
    descriptions.append(longest.lower())

In [6]:
#adds the company descriptions and cleaned descriptions to the dataframe
sp500df["Descriptions"]=descriptions
sp500df["Clean"]=sp500df["Descriptions"].fillna("").str.replace(r"\btable of contents\b"," ",regex=True).str.replace(r"[^a-z\s]", " ",regex=True).str.replace(r"\bitem\s+business\b"," ",regex=True)
for i in sp500df.index:
    temp=sp500df.at[i,"Clean"]
    for w in sp500df.at[i,"Company Name"].lower().split():
        w=re.sub(r"[^a-z]","",w)
        if len(w)>2:
            temp=re.sub(rf"\b{w}\b"," ",temp)
    sp500df.at[i,"Clean"]=temp
sp500df

,Company Name,Ticker,Sector,CIK,url,Descriptions,Clean
0,3M,MMM,Industrials,0000066740,https://www.sec.gov/Archives/edgar/data/66740/...,item 1. business 3m company was incorporated i...,m company was incorporated in under th...
1,A. O. Smith,AOS,Industrials,0000091142,https://www.sec.gov/Archives/edgar/data/91142/...,item 1 – business as used in this annual repor...,as used in this annual report on form k ...
2,Abbott Laboratories,ABT,Health Care,0000001800,https://www.sec.gov/Archives/edgar/data/1800/0...,item 1. business general development of busine...,general development of business is an il...
3,AbbVie,ABBV,Health Care,0001551152,https://www.sec.gov/Archives/edgar/data/155115...,"item 1. business overview abbvie or ""the compa...",overview or the company refer to inc ...
4,Accenture,ACN,Information Technology,0001467373,https://www.sec.gov/Archives/edgar/data/146737...,item 1. business overview accenture is a busin...,overview is a business reinvention and tec...
...,...,...,...,...,...,...,...
495,Xylem Inc.,XYL,Industrials,0001524472,https://www.sec.gov/Archives/edgar/data/152447...,item 1. business business overview xylem is a ...,business overview is a leading global wate...
496,Yum! Brands,YUM,Consumer Discretionary,0001041061,https://www.sec.gov/Archives/edgar/data/104106...,"item 1. business. yum! brands, inc. (referred ...",inc referred to herein as the ...
497,Zebra Technologies,ZBRA,Information Technology,0000877212,https://www.sec.gov/Archives/edgar/data/877212...,item 1. business the company we are a global l...,the company we are a global leader in the au...
498,Zimmer Biomet,ZBH,Health Care,0001136869,https://www.sec.gov/Archives/edgar/data/113686...,item 1. business 4,


In [7]:
#finds top 4 tf-idf words and adds them to the dataframe
matrixvocab=TfidfVectorizer(stop_words="english",min_df=2,max_df=0.8)
tfidf_vocab=matrixvocab.fit_transform(sp500df["Clean"])
counttfidf=matrixvocab.get_feature_names_out()
top4words=[]
for i in range(len(sp500df)):
    toptfidf=pd.Series(tfidf_vocab[i].toarray().ravel(),index=counttfidf)
    top4words.append(", ".join(toptfidf.nlargest(4).index))
sp500df["Top 4 Words"]=top4words
sp500df

,Company Name,Ticker,Sector,CIK,url,Descriptions,Clean,Top 4 Words
0,3M,MMM,Industrials,0000066740,https://www.sec.gov/Archives/edgar/data/66740/...,item 1. business 3m company was incorporated i...,m company was incorporated in under th...,"tapes, president, vice, officer"
1,A. O. Smith,AOS,Industrials,0000091142,https://www.sec.gov/Archives/edgar/data/91142/...,item 1 – business as used in this annual repor...,as used in this annual report on form k ...,"water, heaters, boilers, heater"
2,Abbott Laboratories,ABT,Health Care,0000001800,https://www.sec.gov/Archives/edgar/data/1800/0...,item 1. business general development of busine...,general development of business is an il...,"healthcare, nutritional, treatment, catheters"
3,AbbVie,ABBV,Health Care,0001551152,https://www.sec.gov/Archives/edgar/data/155115...,"item 1. business overview abbvie or ""the compa...",overview or the company refer to inc ...,"fda, approval, patients, european"
4,Accenture,ACN,Information Technology,0001467373,https://www.sec.gov/Archives/edgar/data/146737...,item 1. business overview accenture is a busin...,overview is a business reinvention and tec...,"clients, september, ai, fy"
...,...,...,...,...,...,...,...,...
495,Xylem Inc.,XYL,Industrials,0001524472,https://www.sec.gov/Archives/edgar/data/152447...,item 1. business business overview xylem is a ...,business overview is a leading global wate...,"water, wastewater, customers, solutions"
496,Yum! Brands,YUM,Consumer Discretionary,0001041061,https://www.sec.gov/Archives/edgar/data/104106...,"item 1. business. yum! brands, inc. (referred ...",inc referred to herein as the ...,"restaurant, franchisees, restaurants, franchise"
497,Zebra Technologies,ZBRA,Information Technology,0000877212,https://www.sec.gov/Archives/edgar/data/877212...,item 1. business the company we are a global l...,the company we are a global leader in the au...,"rfid, offerings, barcode, printers"
498,Zimmer Biomet,ZBH,Health Care,0001136869,https://www.sec.gov/Archives/edgar/data/113686...,item 1. business 4,,"aa, aaa, aar, aarp"


In [8]:
#creates csvs of the search volume of each of the four top tf-idf words for each company
googletrendspath=Path("../../QSS20_FA26/finalproject/data")
trendsfolder=googletrendspath/"googletrends"
trendsfolder.mkdir(parents=True,exist_ok=True)
pytrends=TrendReq(hl="en-US",tz=300)
for i in sp500df.index:
    ticker=sp500df.at[i,"Ticker"]
    newfile=trendsfolder/f"{ticker}.csv"
    if newfile.exists():
        continue
    words=sp500df.at[i,"Top 4 Words"].split(", ")
    pytrends.build_payload(words,timeframe="2024-11-01 2026-11-01",geo="US")
    trend=pytrends.interest_over_time()
    trend.drop(columns="isPartial",errors="ignore").to_csv(newfile)
    time.sleep(random.uniform(20,40))

In [9]:
#creates folder path to save plots
plotfolder=googletrendspath/"googletrendsplot"
plotfolder.mkdir(parents=True,exist_ok=True)

In [10]:
#creates folder path to save analysis
analysisfolder=googletrendspath/"analysis"
analysisfolder.mkdir(parents=True,exist_ok=True)

In [11]:
#creates consistent tickers list to be used for future code
tickers=[]
for file in sorted(trendsfolder.glob("*.csv")):
    tickers.append(file.stem.replace(".","-"))
allcloses=yf.download(tickers,start="2024-11-01",end="2026-11-01",interval="1wk",auto_adjust=True,progress=False)["Close"]
allcloses.index=allcloses.index.to_period("W-SAT")

In [12]:
#creates a table for each company displaying the search volume of each of the four tf-idf words, mean of the search volume, and stock trend
for file in sorted(trendsfolder.glob("*.csv")):
    png=plotfolder/f"{file.stem}.png"
    if png.exists():
        continue
    googletrends=pd.read_csv(file,index_col=0,parse_dates=True)
    words=googletrends.columns.tolist()
    googletrends["Mean"]=googletrends.mean(axis=1)
    googletrends["Close"]=allcloses[file.stem.replace(".", "-")].reindex(googletrends.index.to_period("W-SAT")).values
    googletrends["Scaled Close"]=(googletrends["Close"]-googletrends["Close"].min())/(googletrends["Close"].max()-googletrends["Close"].min())*100
    fig,ax=plt.subplots(figsize=(8,3))
    ax.plot(googletrends.index,googletrends["Mean"],linewidth=2.5,color="black",label="mean")
    ax.plot(googletrends.index,googletrends["Scaled Close"],linewidth=2.5,color="red",label="stock")
    for w in words:
        ax.plot(googletrends.index,googletrends[w],linewidth=1,alpha=0.6,label=w)
    ax.set_xlabel("Week")
    ax.set_ylabel("Search interest (0-100)")
    right=ax.secondary_yaxis("right",functions=(lambda s: googletrends["Close"].min()+s/100*(googletrends["Close"].max()-googletrends["Close"].min()),lambda p: (p-googletrends["Close"].min())/(googletrends["Close"].max()-googletrends["Close"].min())*100))
    right.set_ylabel("Stock price ($)")
    plt.title(f"\"{file.stem}\" company mean search interest and stock trends")
    plt.legend()
    plt.tight_layout()
    fig.savefig(png,dpi=150)
    plt.close(fig)

In [13]:
#analyzing the data for correlations
tickers,directionalagreement,pearson,spearman=[],[],[],[]
for file in sorted(trendsfolder.glob("*.csv")):
    ticker=file.stem.replace(".","-")
    if ticker not in allcloses.columns:
        continue
    googletrends=pd.read_csv(file,index_col=0,parse_dates=True)
    googletrends["Mean"]=googletrends.mean(axis=1)
    googletrends["Close"]=allcloses[file.stem.replace(".", "-")].reindex(googletrends.index.to_period("W-SAT")).values
    pricechange=googletrends["Close"].pct_change()
    searchchange=googletrends["Mean"].diff()
    both=pd.concat([pricechange,searchchange],axis=1).dropna()
    both=both[(both["Close"]!=0)&(both["Mean"]!=0)]
    tickers.append(file.stem)
    directionalagreement.append(round((np.sign(both["Close"])==np.sign(both["Mean"])).mean(),3))
    pearson.append(round(abs(both["Close"].corr(both["Mean"])),3))
    spearman.append(round(abs(both["Close"].corr(both["Mean"],method="spearman")),3))
allcompanies_correlations=pd.DataFrame({"Ticker":tickers,"Directional Agreement":directionalagreement,"Pearson":pearson,"Spearman":spearman})
allcompanies_correlations.to_csv(analysisfolder/"correlations.csv",index=False)

In [14]:
#analyzing the data for predictions
tickers,maxlag,lagweeks=[],[],[]
lags=range(-8,9)
for file in sorted(trendsfolder.glob("*.csv")):
    ticker=file.stem.replace(".","-")
    if ticker not in allcloses.columns:
        continue
    googletrends=pd.read_csv(file,index_col=0,parse_dates=True)
    googletrends["Mean"]=googletrends.mean(axis=1)
    googletrends["Close"]=allcloses[file.stem.replace(".", "-")].reindex(googletrends.index.to_period("W-SAT")).values
    pricechange=googletrends["Close"].pct_change()
    searchchange=googletrends["Mean"].diff()
    tickers.append(file.stem)
    results={}
    for lag in lags:
        results[lag]=searchchange.corr(pricechange.shift(-lag),method="spearman")
    lagcorrelations=pd.Series(results)
    maxlag.append(round(lagcorrelations.abs().max(),3))
    lagweeks.append(lagcorrelations.abs().idxmax())
laganalysis=pd.DataFrame({"Ticker":tickers,"Maximum Correlation":maxlag,"Most Correlated Lag (in weeks)":lagweeks})
laganalysis.to_csv(analysisfolder/"lag.csv",index=False)

In [15]:
#results of correlation for all S&P500 companies
print("Directional Agreement:",allcompanies_correlations["Directional Agreement"].mean())
print("Pearson Correlation:",allcompanies_correlations["Pearson"].mean())
print("Spearman Correlation:",allcompanies_correlations["Spearman"].mean())

Directional Agreement: 0.50118
Pearson Correlation: 0.08737200000000002
Spearman Correlation: 0.08538999999999998


In [16]:
#results of prediction for all S&P500 companies
print("Maximum Correlation:",laganalysis["Maximum Correlation"].mean())
print("Most Correlated Lag (in weeks):",laganalysis["Most Correlated Lag (in weeks)"].mean())

Maximum Correlation: 0.21113600000000002
Most Correlated Lag (in weeks): -0.5


In [17]:
#merging sectors with results to make a general dataframe
withSectorCorrelations=allcompanies_correlations.merge(sp500df[["Ticker","Sector"]],on="Ticker",how="left")
withSectorLag=laganalysis.merge(sp500df[["Ticker","Sector"]],on="Ticker",how="left")
sectoranalysis=pd.DataFrame({"Directional Agreement":round(withSectorCorrelations.groupby("Sector")["Directional Agreement"].mean(),3),"Pearson":round(withSectorCorrelations.groupby("Sector")["Pearson"].mean(),3),"Spearman":round(withSectorCorrelations.groupby("Sector")["Spearman"].mean(),3),"Predictive Correlation":round(withSectorLag.groupby("Sector")["Maximum Correlation"].mean(),3),"Best Predictive Week":round(withSectorLag.groupby("Sector")["Most Correlated Lag (in weeks)"].mean(),3)})
sectoranalysis.to_csv(analysisfolder/"sectors.csv",index=False)

In [18]:
#results of correlation and prediction, sector by sector
print("Directional Agreement:",sectoranalysis["Directional Agreement"].abs().idxmax())
print("Pearson Correlation:",sectoranalysis["Pearson"].abs().idxmax())
print("Spearman Correlation:",sectoranalysis["Spearman"].abs().idxmax())
print("Predictive Correlation:",sectoranalysis["Predictive Correlation"].abs().idxmax())

Directional Agreement: Consumer Discretionary
Pearson Correlation: Consumer Staples
Spearman Correlation: Consumer Staples
Predictive Correlation: Consumer Staples
